# Lab 02 — Medallion Architecture: Pandas & Parquet on S3

This notebook walks through the **Medallion Architecture** (Bronze → Silver → Gold)
using Pandas and Parquet on RustFS S3 storage.

| Layer | Purpose | Format |
|-------|---------|--------|
| **Bronze** | Raw data, ingested exactly as received | CSV |
| **Silver** | Cleaned, filtered, enriched | Parquet |
| **Gold** | Aggregated, business-ready analytics | Parquet |

### Why Parquet for Silver and Gold?

Parquet is a **columnar binary format** optimized for analytical workloads:
- **Compression**: 3-10× smaller than CSV for typical datasets
- **Column pruning**: queries read only the columns they need
- **Typed schema**: no type inference surprises at read time
- **Ecosystem standard**: native in Spark, DuckDB, Pandas, and all major cloud query engines


---
## 0 · Prerequisites

Before running this notebook, make sure:

1. RustFS is running: `docker compose up -d`
2. Buckets `bronze`, `silver`, and `gold` exist (created by `rustfs-init`)
3. The virtual environment is active with dependencies installed: `uv sync`
   - `boto3`, `pandas`, `pyarrow` are required


---
## Setup — Connect and Import Libraries

We connect using `Config(signature_version='s3v4')` — required for RustFS and other
non-AWS S3-compatible stores. Without it, boto3 generates Signature V2 URLs that
RustFS rejects.


In [ ]:
import os
import tempfile

import boto3
import pandas as pd
from botocore.config import Config
from IPython.display import display

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

# Shared temp directory for staging files between cells
tmpdir = tempfile.mkdtemp()

print('✅ Connected to RustFS S3!')
print(f'   Staging directory: {tmpdir}')


---
## 1 · Bronze Layer — Raw Ingestion

The Bronze layer stores **data exactly as it arrives** from the source — no filtering,
no cleaning. This is your audit trail: if a downstream transformation has a bug,
you can always reprocess from Bronze.

Here we create a synthetic e-commerce orders dataset and upload it as raw CSV. We'll
also attach a small `Metadata` dict — via `ExtraArgs` on `upload_file` — describing the
ingestion (source, row count, schema version). It travels in the same upload call, so
there's no separate `metadata.json` to keep in sync with the CSV.


In [ ]:
# Synthetic orders dataset — raw, with mixed statuses
orders_raw = pd.DataFrame({
    'order_id':    [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    'customer_id': ['C001','C002','C003','C004','C005','C006','C007','C008','C009','C010'],
    'product':     ['Laptop','Monitor','Keyboard','Mouse','Webcam',
                    'Headset','Tablet','Charger','Phone Case','USB Hub'],
    'amount':      [1299.0, 349.0, 89.90, 29.99, 149.0, 199.0, 499.0, 39.99, 19.99, 49.99],
    'status':      ['PAID','PAID','CANCELLED','PAID','REFUNDED',
                    'PAID','CANCELLED','PAID','PAID','REFUNDED'],
    'order_date':  ['2025-01-10','2025-01-11','2025-01-12','2025-01-13','2025-01-14',
                    '2025-01-15','2025-01-16','2025-01-17','2025-01-18','2025-01-19'],
})

print('📋 Raw orders (Bronze):')
display(orders_raw)

# Upload raw CSV to Bronze via a local staging file, metadata in the same call
local_csv = os.path.join(tmpdir, 'orders_raw.csv')
orders_raw.to_csv(local_csv, index=False)
s3.upload_file(
    local_csv, 'bronze', 'sales/orders_raw.csv',
    ExtraArgs={
        'Metadata': {
            'source':         'synthetic-orders-generator',
            'rows':           str(len(orders_raw)),
            'schema_version': '1.0',
        }
    },
)

print('\n✅ Ingested to Bronze: s3://bronze/sales/orders_raw.csv')
print(f'   {len(orders_raw)} rows | {os.path.getsize(local_csv):,} bytes')


---
## 2 · Silver Layer — Clean & Transform

Silver reads from Bronze and applies **business rules**:
- Keep only `PAID` orders (exclude cancelled and refunded)
- Parse `order_date` as a proper datetime
- Add a derived `tax` column (10% of amount)

The result is saved as **Parquet** — a much more efficient format for downstream queries.
Just like Bronze, the Silver upload attaches a `Metadata` dict (source layer, row counts
before/after, the filter rule applied) in the same `upload_file` call — no separate
quality-report file to maintain.


In [ ]:
# Download raw CSV from Bronze
local_csv = os.path.join(tmpdir, 'orders_raw.csv')
s3.download_file('bronze', 'sales/orders_raw.csv', local_csv)
orders_raw = pd.read_csv(local_csv)

# --- Cleaning steps ---
# 1. Filter to paid orders only
orders_clean = orders_raw[orders_raw['status'] == 'PAID'].copy()

# 2. Parse order_date as datetime
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'])

# 3. Add tax column
orders_clean['tax'] = (orders_clean['amount'] * 0.10).round(2)

print(f'📋 Cleaned orders (Silver): {len(orders_raw)} → {len(orders_clean)} rows'
      f' ({len(orders_raw) - len(orders_clean)} removed)')
display(orders_clean)

# Upload Parquet to Silver, metadata (row counts, filter rule) in the same call
local_parquet = os.path.join(tmpdir, 'orders_clean.parquet')
orders_clean.to_parquet(local_parquet, index=False, engine='pyarrow')
s3.upload_file(
    local_parquet, 'silver', 'sales/orders_clean.parquet',
    ExtraArgs={
        'Metadata': {
            'source_layer':  'bronze',
            'rows_in':       str(len(orders_raw)),
            'rows_out':      str(len(orders_clean)),
            'rows_removed':  str(len(orders_raw) - len(orders_clean)),
            'filter_rule':   'status == PAID',
        }
    },
)

csv_size = os.path.getsize(local_csv)
parquet_size = os.path.getsize(local_parquet)
print('\n✅ Uploaded to Silver: s3://silver/sales/orders_clean.parquet')
print(f'   CSV size: {csv_size:,} bytes → Parquet size: {parquet_size:,} bytes '
      f'({parquet_size/csv_size:.0%} of original)')


---
## 3 · Gold Layer — Aggregation

Gold reads from Silver and produces **business-ready summaries**.
Here we aggregate revenue and order count by product, sorted by total revenue.

Gold tables are the ones consumed by BI dashboards (Power BI, Tableau, Grafana)
and executive reports — they should be fast to read and easy to interpret. The
Gold upload also carries its own `Metadata` (source layer, KPI name, row count) —
the same pattern used for Bronze and Silver.


In [ ]:
# Download Parquet from Silver
local_parquet = os.path.join(tmpdir, 'orders_clean.parquet')
s3.download_file('silver', 'sales/orders_clean.parquet', local_parquet)
orders_clean = pd.read_parquet(local_parquet)

# Aggregate: revenue and count per product
orders_agg = (
    orders_clean
    .groupby('product')
    .agg(
        total_revenue=('amount', 'sum'),
        order_count=('order_id', 'count'),
        avg_ticket=('amount', 'mean'),
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

print('📊 Revenue by product (Gold):')
display(orders_agg)

# Upload aggregated Parquet to Gold, metadata in the same call
local_agg = os.path.join(tmpdir, 'orders_agg.parquet')
orders_agg.to_parquet(local_agg, index=False, engine='pyarrow')
s3.upload_file(
    local_agg, 'gold', 'sales/orders_agg.parquet',
    ExtraArgs={
        'Metadata': {
            'source_layer': 'silver',
            'kpi':          'revenue_by_product',
            'rows':         str(len(orders_agg)),
        }
    },
)

print('\n✅ Uploaded to Gold: s3://gold/sales/orders_agg.parquet')


---
## 4 · Inspect All Layers

Let's verify the full pipeline by listing every object across the three buckets. For
each object we also call `head_object` to preview its custom `Metadata` — proof that
provenance and quality info travel with the file itself, with no `.json` sidecar
anywhere in the bucket.


In [ ]:
print('🗂️  Medallion Architecture — Object Inventory\n')
for bucket in ['bronze', 'silver', 'gold']:
    resp = s3.list_objects_v2(Bucket=bucket, Prefix='sales/')
    objects = resp.get('Contents', [])
    total_bytes = sum(o['Size'] for o in objects)
    print(f's3://{bucket}/  ({len(objects)} object(s), {total_bytes:,} bytes total)')
    for obj in objects:
        head = s3.head_object(Bucket=bucket, Key=obj['Key'])
        meta_preview = ', '.join(f'{k}={v}' for k, v in head['Metadata'].items())
        print(f'  └─ {obj["Key"]:35s}  {obj["Size"]:>8,} bytes  [{meta_preview}]')
    print()

print('✅ Medallion pipeline complete! No metadata.json anywhere — every object carries its own.')


---
## 5 · Cleanup

Remove the objects created during this lab and the local temp files.


In [ ]:
import shutil

keys_to_delete = {
    'bronze': 'sales/orders_raw.csv',
    'silver': 'sales/orders_clean.parquet',
    'gold':   'sales/orders_agg.parquet',
}

for bucket, key in keys_to_delete.items():
    s3.delete_object(Bucket=bucket, Key=key)
    print(f'🗑️  Deleted s3://{bucket}/{key}')

shutil.rmtree(tmpdir, ignore_errors=True)
print(f'🗑️  Removed staging directory: {tmpdir}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Layer | S3 Path | Format | What changed |
|-------|---------|--------|-------------|
| Bronze | `s3://bronze/sales/orders_raw.csv` | CSV | Nothing — raw ingestion |
| Silver | `s3://silver/sales/orders_clean.parquet` | Parquet | Filtered (PAID only), typed dates, added tax |
| Gold | `s3://gold/sales/orders_agg.parquet` | Parquet | Aggregated by product |

### Key takeaways

- **Bronze** is immutable — always keep the raw source for reprocessing
- **Silver** is where data quality is enforced — schema, types, business rules
- **Gold** is optimized for fast reads by analysts and BI tools
- **Parquet** reduces file size significantly vs CSV and enables column pruning in query engines
- **Metadata rides with the object** — every upload in this lab passed a `Metadata` dict via `ExtraArgs`, so provenance and lineage info is always one `head_object` call away, with no sidecar file to drift out of sync

### Next steps

- **Lab 03**: Multipart upload for large files (> 100 MB)
- **Lab 05**: Object versioning — track changes to Silver/Gold tables over time
- **Lab 11**: The full production-grade version of this pipeline, with quality reports and partitioning
